# Chapter 7: Working with Nested Data Structures

## Technical Requirements

In [1]:
import polars as pl

In [3]:
df = pl.read_csv("../data/us_videos.csv", try_parse_dates=True)
df.glimpse(max_items_per_column=2)

Rows: 40949
Columns: 16
$ video_id                             <str> '2kyS6SvSYSE', '1ZAPwfrtAFY'
$ trending_date                        <str> '17.14.11', '17.14.11'
$ title                                <str> 'WE WANT TO TALK ABOUT OUR MARRIAGE', 'The Trump Presidency: Last Week Tonight with John Oliver (HBO)'
$ channel_title                        <str> 'CaseyNeistat', 'LastWeekTonight'
$ category_id                          <i64> 22, 24
$ publish_time           <datetime[μs, UTC]> 2017-11-13 17:13:01+00:00, 2017-11-13 07:30:00+00:00
$ tags                                 <str> 'SHANtell martin', 'last week tonight trump presidency|last week tonight donald trump|john oliver trump|donald trump'
$ views                                <i64> 748374, 2418783
$ likes                                <i64> 57527, 97185
$ dislikes                             <i64> 2966, 6146
$ comment_count                        <i64> 15954, 12703
$ thumbnail_link                       <str> 'https://i.ytimg

In [ ]:
df.head()

In [4]:
df = df.with_columns(pl.col("trending_date").str.to_date(format="%y.%d.%m"))

In [5]:
df.select('trending_date').dtypes[0]

Date

## Creating lists

### How to do it...

In [6]:
df.select(
    'tags',
    pl.col('tags').str.split('|').alias('tags in list')
).head()    

tags,tags in list
str,list[str]
"""SHANtell martin""","[""SHANtell martin""]"
"""last week tonight trump presid…","[""last week tonight trump presidency"", ""last week tonight donald trump"", … ""donald trump""]"
"""racist superman|rudy|mancuso|k…","[""racist superman"", ""rudy"", … "" Lele Pons""]"
"""rhett and link|gmm|good mythic…","[""rhett and link"", ""gmm"", … ""challenge""]"
"""ryan|higa|higatv|nigahiga|i da…","[""ryan"", ""higa"", … ""fail""]"


In [7]:
(
    df
    .group_by('trending_date')
    .agg(pl.col('video_id'))
    .sort('trending_date', descending=True)
).head()

trending_date,video_id
date,list[str]
2018-06-14,"[""-QPdRfqTnt4"", ""gPHVLxm8U-0"", … ""ooyjaVdt-jA""]"
2018-06-13,"[""FchkqXEg0qs"", ""uHRwMmwbFnA"", … ""Q5KmA3Xbmqo""]"
2018-06-12,"[""PPWDwBrUNyY"", ""rAH8qm5oQHg"", … ""6S9c5nnDd_s""]"
2018-06-11,"[""0bXCbVGb04A"", ""L4pkD78oKSo"", … ""r-3iathMo7o""]"
2018-06-10,"[""L4pkD78oKSo"", ""ZFwylDNpgFc"", … ""r-3iathMo7o""]"


In [8]:
df.select(
    pl.concat_list(
        pl.col('views'),
        pl.col('likes'),
        pl.col('dislikes'),
        pl.col('comment_count')
    ).alias('engagement')
).head()

engagement
list[i64]
"[748374, 57527, … 15954]"
"[2418783, 97185, … 12703]"
"[3191434, 146033, … 8181]"
"[343168, 10172, … 2146]"
"[2095731, 132235, … 17518]"


### There is more...

In [ ]:
df = pl.DataFrame({
    'nested_list': [
        [
            [1,2,3], [4,5,6], 
            [7,8,9], [10,11,12]
        ], 
        [
            [1,2,3], [4,5,6], 
            [7,8,9], [10,11,12]
        ]
    ]
})

In [ ]:
df

In [ ]:
df = pl.DataFrame({
    'nested_list': [
        [
            ['a',2,3], [4,5,6], 
            [7,8,9], [10,11,12]
        ], 
        [
            [1,2,3], [4,5,6], 
            [7,8,9], [10,11,12]
        ]
    ]
}, strict=False)

In [ ]:
df

## Aggregating elements in lists

### How to do it...

In [ ]:
import polars as pl

In [9]:
df = (
    pl.scan_csv('../data/us_videos.csv', try_parse_dates=True)
    .with_columns(
        pl.col('trending_date').str.to_date(format='%y.%d.%m')
    ).collect()
)
df.head()

video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
str,date,str,str,i64,"datetime[μs, UTC]",str,i64,i64,i64,i64,str,bool,bool,bool,str
"""2kyS6SvSYSE""",2017-11-14,"""WE WANT TO TALK ABOUT OUR MARR…","""CaseyNeistat""",22,2017-11-13 17:13:01 UTC,"""SHANtell martin""",748374,57527,2966,15954,"""https://i.ytimg.com/vi/2kyS6Sv…",false,false,false,"""SHANTELL'S CHANNEL - https://w…"
"""1ZAPwfrtAFY""",2017-11-14,"""The Trump Presidency: Last Wee…","""LastWeekTonight""",24,2017-11-13 07:30:00 UTC,"""last week tonight trump presid…",2418783,97185,6146,12703,"""https://i.ytimg.com/vi/1ZAPwfr…",false,false,false,"""One year after the presidentia…"
"""5qpjK5DgCt4""",2017-11-14,"""Racist Superman | Rudy Mancuso…","""Rudy Mancuso""",23,2017-11-12 19:05:24 UTC,"""racist superman|rudy|mancuso|k…",3191434,146033,5339,8181,"""https://i.ytimg.com/vi/5qpjK5D…",false,false,false,"""WATCH MY PREVIOUS VIDEO ▶ \n\n…"
"""puqaWrEC7tY""",2017-11-14,"""Nickelback Lyrics: Real or Fak…","""Good Mythical Morning""",24,2017-11-13 11:00:04 UTC,"""rhett and link|gmm|good mythic…",343168,10172,666,2146,"""https://i.ytimg.com/vi/puqaWrE…",false,false,false,"""Today we find out if Link is a…"
"""d380meD0W0M""",2017-11-14,"""I Dare You: GOING BALD!?""","""nigahiga""",24,2017-11-12 18:01:41 UTC,"""ryan|higa|higatv|nigahiga|i da…",2095731,132235,1989,17518,"""https://i.ytimg.com/vi/d380meD…",false,false,false,"""I know it's been a while since…"


In [10]:
agg_df = (
    df
    .group_by('trending_date')
    .agg(
        'views',
        'likes',
        'dislikes',
        'comment_count'
    )
    .sort('trending_date', descending=True)
)
agg_df.head()

trending_date,views,likes,dislikes,comment_count
date,list[i64],list[i64],list[i64],list[i64]
2018-06-14,"[4427381, 5829270, … 10306119]","[96391, 87323, … 357079]","[5508, 3668, … 212976]","[12726, 11933, … 144795]"
2018-06-13,"[3238183, 470844, … 7839668]","[61841, 13922, … 352352]","[3708, 402, … 5871]","[0, 4843, … 46624]"
2018-06-12,"[3483553, 6173038, … 13619534]","[23725, 90478, … 347100]","[3145, 3877, … 6923]","[462, 7726, … 19977]"
2018-06-11,"[2341772, 846887, … 6995168]","[140374, 5758, … 143678]","[2951, 850, … 10925]","[33760, 1539, … 17444]"
2018-06-10,"[594004, 1504564, … 6980540]","[4470, 32430, … 143452]","[520, 4316, … 10911]","[1195, 6105, … 17429]"


In [ ]:
df_pd = df.to_pandas()

In [ ]:
agg_df_pd = (
    df_pd.groupby('trending_date').agg(
        {'views': list, 'likes': list, 'dislikes': list, 'comment_count': list}
    ).sort_values('trending_date', ascending=False)
)

In [ ]:
agg_df_pd.transform({'views': len}).rename(columns={'views': 'views_count'})

In [ ]:
(
    agg_df
    .select(
        'trending_date',
        pl.col('views').list.min().alias('views_min'),
        pl.col('likes').list.max().alias('likes_max'),
        pl.col('dislikes').list.mean().alias('dislikes_mean'),
        pl.col('comment_count').list.sum().alias('comment_sum'),
    )
    .sort('trending_date', descending=True)
).head()

In [ ]:
(
    df
    .group_by('trending_date')
    .agg(pl.col('channel_title'))
    .with_columns(
        pl.col('channel_title').list.join(':')
    )
    .sort('trending_date', descending=True)
).head()

### There is more...

In [ ]:
(
    agg_df
    .select(
        'trending_date',
        pl.col('views').list.len().alias('item_cnt')
    )
).head()

## Accessing and selecting elements in lists

### Getting ready

In [ ]:
import polars as pl

In [ ]:
trending_dates_by_channel = (
    df
    .group_by('channel_title')
    .agg('trending_date')
    .with_columns(pl.col('trending_date').list.sort())
)

In [ ]:
trending_dates_by_channel.head()

### How to do it...

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col('trending_date').list.first().alias('first_trending_date'),
    pl.col('trending_date').list.last().alias('last_trending_date')
).head()

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col('trending_date').list.get(7, null_on_oob=True).alias('8th_element')
).head()

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col('trending_date').list.head().alias('first_5'),
    pl.col('trending_date').list.tail(10).alias('last_10')
).head()

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col('trending_date')
    .list.sort(descending=True)
    .list.head(3)
    .alias('3_most_recent_dates')
).head()

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col('trending_date')
    .list.sort(descending=True)
    .list.head(3)
    .alias('3_most_recent_dates')
).head()

In [ ]:
trending_dates_by_channel.select(
    'trending_date',
    pl.col('trending_date').list.slice(0, 2).alias('first_2_dates'),
    pl.col('trending_date').list.slice(-3, 1).alias('3rd_date_to_last'),
    pl.col('trending_date').list.slice(7).alias('from_8th_date_to_end')
).head()

In [ ]:
trending_dates_by_channel.select(
    'trending_date',
    pl.col('trending_date').list.gather([0, -1]).alias('first_and_last'),
    pl.col('trending_date').list.gather([0, 0, 0, 0]).alias('first_repeated'),
    pl.col('trending_date').list.gather([0, 10], null_on_oob=True).alias('first_and_10th_or_null'),
).head()

### There is more...

In [ ]:
(
    df
    .group_by('trending_date')
    .agg('category_id')
    .with_columns(pl.col('category_id').list.sort())
    .with_columns(
        pl.col('category_id'),
        pl.col('category_id').list.len().alias('category_id_cnt'),
        pl.col('category_id').list.unique().alias('category_id_unique'),
        pl.col('category_id').list.unique().list.len().alias('category_id_unique_cnt')
    )
).head()

In [ ]:
trending_dates_by_channel.with_columns(
    pl.col("trending_date")
    .list.sample(n=3, with_replacement=True, seed=0)
    .alias("samples")
).head()

## Applying logic to each element in lists

### Getting ready

In [ ]:
agg_df = (
    df
    .group_by('trending_date')
    .agg('views', 'channel_title')
)

In [ ]:
agg_df.head()

### How to do it...

In [ ]:
(
    agg_df
    .select(
        'views',
        pl.col('views').list.eval(pl.element()).alias('pl.element'),
        pl.col('views').list.eval(pl.element().first()).alias('pl.first'),
        pl.col('views').list.eval(pl.element().last()).alias('pl.last'),
        pl.col('views').list.eval(pl.element()).alias('pl.col')
    )
).head()

In [ ]:
channel_titles_df = (
    agg_df
    .select(
        pl.col("channel_title").list.head(2),
        pl.col("channel_title")
        .list.eval(pl.element().str.to_uppercase())
        .list.head(2)
        .alias("channel_title_upper")
    )
)
channel_titles_df.head()

In [ ]:
(
    channel_titles_df
    .with_columns(
        pl.col('channel_title_upper')
        .list.eval(
            pl.element().filter(
                pl.element().str.contains('A', literal=True)
            )
        )
    )
).head()

In [ ]:
(
    agg_df
    .select(
        'trending_date',
        'views',
        pl.col('views')
        .list.eval(pl.element().rank('dense', descending=True))
        .alias('views_rank'),
    )
).head()

In [ ]:
views_rank_df = (
    agg_df
    .select(
        'trending_date',
        'views',
        pl.col('views')
        .list.eval(pl.element().rank('dense', descending=True))
        .alias('views_rank'),
    )
)
views_rank_df.head()

In [ ]:
top3_views_df = (
    views_rank_df
    .explode('views', 'views_rank')
    .filter(pl.col('views_rank')<=3)
    .group_by('trending_date')
    .agg(pl.all())
)
top3_views_df.head()

In [ ]:
(
    views_rank_df
    .explode('views', 'views_rank')
).head()

In [ ]:
(
    top3_views_df
    .select(
        'views_rank',
        'views',
        pl.col('views')
        .list.eval(pl.element().max() - pl.element())
        .alias('diff from the most views')
    )
).head()

### There is more...

In [ ]:
top3_views_df.head()

In [ ]:
(
    top3_views_df
    .with_columns(
        pl.col('views_rank').list.slice(0, 2).alias('views_rank_1'),
        pl.col('views_rank').list.slice(-2, 2).alias('views_rank_2')
    )
    .select(
        'views_rank_1',
        'views_rank_2',
        pl.col('views_rank_1').list.set_intersection('views_rank_2')
        .alias('intersection'),
        pl.col('views_rank_1').list.set_union('views_rank_2')
        .alias('union'),
        pl.col('views_rank_1').list.set_difference('views_rank_2')
        .alias('difference'),
        pl.col('views_rank_1').list.set_symmetric_difference('views_rank_2')
        .alias('symmetric_difference'),
    )
).head()

## Working with structs and JSON data

### Getting ready

In [ ]:
import polars as pl

In [20]:
df = pl.read_json('../data/ga_20170801.json', infer_schema_length=1_000)
df.head()

visitorId,visitNumber,visitId,visitStartTime,date,totals,trafficSource,device,geoNetwork,customDimensions,hits,fullVisitorId,userId,clientId,channelGrouping,socialEngagementType
null,str,str,str,str,struct[13],struct[9],struct[17],struct[11],list[struct[2]],list[struct[33]],str,null,null,str,str
null,"""1""","""1501591568""","""1501591568""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{null,""(not set)"",""(direct)"",""(none)"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","{""Chrome"",""not available in demo dataset"",""not available in demo dataset"",""Windows"",""not available in demo dataset"",""false"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",null,""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""desktop""}","{""Europe"",""Southern Europe"",""Greece"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""tellas.gr"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset""}",[],"[{[],[],null,""0"",null,""web"",{null,""true"",null,null},""46"",""true"",""true"",null,{""/google+redesign/bags/google+zipper+front+sports+bag.axd"",""shop.googlemerchandisestore.com"",""Page Unavailable"",null,null,""/google+redesign/"",""/bags/"",""/google+zipper+front+sports+bag.axd"",""""},""1"",null,null,null,null,null,null,{""0"",""1"",null},""https://www.google.gr/"",[],[],""PAGE"",[],null,""5"",""true"",[],{null,null,null,null,""(not set)"",null,""No"","" : ""},{""(not set)"",""Bags"",""(not set)"",""(not set)"",""(not set)"",""(entrance)"",""(entrance)"",""(entrance)"",""(entrance)"",""(entrance)"",null,""1"",null,null,null},[],{null,null,null,null,null,null,null,null,""shop.googlemerchandisestore.com/google+redesign/bags/google+zipper+front+sports+bag.axd"",""shop.googlemerchandisestore.com/google+redesign/bags/google+zipper+front+sports+bag.axd"",""shop.googlemerchandisestore.com/google+redesign/bags/google+zipper+front+sports+bag.axd"",""0""}}]","""3418334011779872055""",null,null,"""Organic Search""","""Not Socially Engaged"""
null,"""2""","""1501589647""","""1501589647""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,null,null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","{""Chrome"",""not available in demo dataset"",""not available in demo dataset"",""Windows"",""not available in demo dataset"",""false"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",null,""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset"",""desktop""}","{""Asia"",""Southern Asia"",""India"",""Maharashtra"",""(not set)"",""Mumbai"",""not available in demo dataset"",""unknown.unknown"",""not available in demo dataset"",""not available in demo dataset"",""not available in demo dataset""}","[{""4"",""APAC""}]","[{[],[],null,""0"",null,""web"",{null,""true"",null,null},""14"",""true"",""true"",null,{""/google+redesign/shop+by+brand/youtube"",""shop.googlemerchandisestore.com"",""Page Unavailable"",null,null,""/google+redesign/"",""/shop+by+brand/"",""/youtube"",""""},""1"",null,null,null,null,null,null,{""0"",""1"",null},""https://analytics.google.com/analytics/web/"",[],[],""PAGE"",[],null,""5"",""true"",[],{null,null,null,null,""(not set)"",null,""No"","" : ""},{""(not set)"",""Brands"",""(not set)"",""(not set)"",""(not set)"",""(entrance)""

In [21]:
cols = ['visitId', 'date', 'totals', 'trafficSource', 'customDimensions', 'channelGrouping']
df = df.select(cols)
df.head()

visitId,date,totals,trafficSource,customDimensions,channelGrouping
str,str,struct[13],struct[9],list[struct[2]],str
"""1501591568""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{null,""(not set)"",""(direct)"",""(none)"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}",[],"""Organic Search"""
"""1501589647""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,null,null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""APAC""}]","""Referral"""
"""1501616621""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""EMEA""}]","""Referral"""
"""1501601200""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""North America""}]","""Referral"""
"""1501615525""","""20170801""","{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""adwords.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""North America""}]","""Referral"""


In [22]:
df.select('totals', 'trafficSource', 'customDimensions').head()

totals,trafficSource,customDimensions
struct[13],struct[9],list[struct[2]]
"{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{null,""(not set)"",""(direct)"",""(none)"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}",[]
"{""1"",""1"",""1"",null,""1"",null,null,null,null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""APAC""}]"
"{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""EMEA""}]"
"{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""analytics.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""North America""}]"
"{""1"",""1"",""1"",null,""1"",null,null,""1"",null,null,null,null,""1""}","{""/analytics/web/"",""(not set)"",""adwords.google.com"",""referral"",null,null,{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null},null,null}","[{""4"",""North America""}]"


### How to do it...

In [23]:
df = df.with_columns(
    pl.struct('visitId', 'date', 'channelGrouping').alias('structFromCols')
)

In [24]:
(
    df
    .select(
        'visitId',
        'date',
        'channelGrouping',
        pl.struct('visitId', 'date', 'channelGrouping').alias('structFromCols')
    )
).head()

visitId,date,channelGrouping,structFromCols
str,str,str,struct[3]
"""1501591568""","""20170801""","""Organic Search""","{""1501591568"",""20170801"",""Organic Search""}"
"""1501589647""","""20170801""","""Referral""","{""1501589647"",""20170801"",""Referral""}"
"""1501616621""","""20170801""","""Referral""","{""1501616621"",""20170801"",""Referral""}"
"""1501601200""","""20170801""","""Referral""","{""1501601200"",""20170801"",""Referral""}"
"""1501615525""","""20170801""","""Referral""","{""1501615525"",""20170801"",""Referral""}"


In [26]:
(
    df
    .group_by('channelGrouping')
    .agg(
        'visitId', 
        pl.col('visitId').len().alias('numVisits')
    )
    .sort('numVisits')
    .with_columns(
        pl.col('visitId').list.to_struct(['visit_1', 'visit_2']).alias('struct_from_list')
    )   
)

channelGrouping,visitId,numVisits,struct_from_list
str,list[str],u32,struct[2]
"""Display""","[""1501651856"", ""1501625928"", … ""1501638116""]",12,"{""1501651856"",""1501625928""}"
"""Paid Search""","[""1501610896"", ""1501644116"", … ""1501613648""]",20,"{""1501610896"",""1501644116""}"
"""Affiliates""","[""1501604627"", ""1501572101"", … ""1501635918""]",29,"{""1501604627"",""1501572101""}"
"""Referral""","[""1501589647"", ""1501616621"", … ""1501607798""]",106,"{""1501589647"",""1501616621""}"
"""Social""","[""1501590147"", ""1501655923"", … ""1501652602""]",136,"{""1501590147"",""1501655923""}"
"""Direct""","[""1501586309"", ""1501587435"", … ""1501610792""]",163,"{""1501586309"",""1501587435""}"
"""Organic Search""","[""1501591568"", ""1501583103"", … ""1501625964""]",534,"{""1501591568"",""1501583103""}"


In [ ]:
(
    df
    .select(
        'structFromCols',
        pl.col('structFromCols').alias('structFromColsToBeUnpacked')
    )
    .unnest('structFromColsToBeUnpacked')
).head()

structFromCols,visitId,date,channelGrouping
struct[3],str,str,str
"{""1501591568"",""20170801"",""Organic Search""}","""1501591568""","""20170801""","""Organic Search"""
"{""1501589647"",""20170801"",""Referral""}","""1501589647""","""20170801""","""Referral"""
"{""1501616621"",""20170801"",""Referral""}","""1501616621""","""20170801""","""Referral"""
"{""1501601200"",""20170801"",""Referral""}","""1501601200""","""20170801""","""Referral"""
"{""1501615525"",""20170801"",""Referral""}","""1501615525""","""20170801""","""Referral"""


In [28]:
(
    df
    .select(
        pl.col('trafficSource')
    )
    .unnest('trafficSource')
).head()

referralPath,campaign,source,medium,keyword,adContent,adwordsClickInfo,isTrueDirect,campaignCode
str,str,str,str,str,str,struct[12],str,null
null,"""(not set)""","""(direct)""","""(none)""",null,null,"{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null}",null,null
"""/analytics/web/""","""(not set)""","""analytics.google.com""","""referral""",null,null,"{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null}",null,null
"""/analytics/web/""","""(not set)""","""analytics.google.com""","""referral""",null,null,"{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null}",null,null
"""/analytics/web/""","""(not set)""","""analytics.google.com""","""referral""",null,null,"{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null}",null,null
"""/analytics/web/""","""(not set)""","""adwords.google.com""","""referral""",null,null,"{null,null,null,null,null,null,""not available in demo dataset"",null,null,null,null,null}",null,null


In [29]:
(
    df
    .select(
        'structFromCols',
        pl.col('structFromCols').struct.rename_fields(['a', 'b', 'c']).alias('renamedStructToBeUnpacked')
    )
    .unnest('renamedStructToBeUnpacked')
).head()

structFromCols,a,b,c
struct[3],str,str,str
"{""1501591568"",""20170801"",""Organic Search""}","""1501591568""","""20170801""","""Organic Search"""
"{""1501589647"",""20170801"",""Referral""}","""1501589647""","""20170801""","""Referral"""
"{""1501616621"",""20170801"",""Referral""}","""1501616621""","""20170801""","""Referral"""
"{""1501601200"",""20170801"",""Referral""}","""1501601200""","""20170801""","""Referral"""
"{""1501615525"",""20170801"",""Referral""}","""1501615525""","""20170801""","""Referral"""


In [30]:
(
    df
    .select(
        'structFromCols',
        pl.col('structFromCols').struct.field('channelGrouping')
    )
).head()

structFromCols,channelGrouping
struct[3],str
"{""1501591568"",""20170801"",""Organic Search""}","""Organic Search"""
"{""1501589647"",""20170801"",""Referral""}","""Referral"""
"{""1501616621"",""20170801"",""Referral""}","""Referral"""
"{""1501601200"",""20170801"",""Referral""}","""Referral"""
"{""1501615525"",""20170801"",""Referral""}","""Referral"""


In [31]:
(
    df
    .select(
        pl.struct(
            pl.col('channelGrouping'),
            pl.col('trafficSource').struct.field('source')
        )
        .unique()
        .alias('channelAndSource')
    )
    .unnest('channelAndSource')
    .sort('channelGrouping', 'source')
)

channelGrouping,source
str,str
"""Affiliates""","""Partners"""
"""Direct""","""(direct)"""
"""Display""","""(direct)"""
"""Display""","""dfa"""
"""Organic Search""","""(direct)"""
…,…
"""Social""","""groups.google.com"""
"""Social""","""l.facebook.com"""
"""Social""","""m.facebook.com"""


### There is more...

In [42]:
total_struct_to_str_expr = pl.col('totals').struct.json_encode()
(
    df
    .select(
        total_struct_to_str_expr.alias('total_str'),
        total_struct_to_str_expr
        .str.json_decode(dtype=pl.Struct({
            'visits': pl.String,
            'hits': pl.String,
            'pageviews': pl.String,
            'bounces': pl.String,
        }))
        .alias('total_struct')
    )
).head()

total_str,total_struct
str,struct[4]
"""{""visits"":""1"",""hits"":""1"",""page…","{""1"",""1"",""1"",""1""}"
"""{""visits"":""1"",""hits"":""1"",""page…","{""1"",""1"",""1"",""1""}"
"""{""visits"":""1"",""hits"":""1"",""page…","{""1"",""1"",""1"",""1""}"
"""{""visits"":""1"",""hits"":""1"",""page…","{""1"",""1"",""1"",""1""}"
"""{""visits"":""1"",""hits"":""1"",""page…","{""1"",""1"",""1"",""1""}"


In [44]:
df.select(total_struct_to_str_expr.alias('total_str')).item(0,0)

'{"visits":"1","hits":"1","pageviews":"1","timeOnSite":null,"bounces":"1","transactions":null,"transactionRevenue":null,"newVisits":"1","screenviews":null,"uniqueScreenviews":null,"timeOnScreen":null,"totalTransactionRevenue":null,"sessionQualityDim":"1"}'